# Query a Foundry IQ knowledge base through MCP

Connect directly to the Azure AI Search MCP endpoint for the minimal-reasoning `contoso-company-kb-minimal` knowledge base, inspect the tool it exposes, and retrieve grounded HR and benefits content. The endpoint uses the signed-in Azure Developer CLI identity and requires Search data-plane access.

## Step 1: Load configuration and create an MCP session helper

Load the Azure tenant, Search endpoint, and minimal knowledge-base name from the project `.env` file. The MCP URL targets the knowledge base directly and uses an Azure AI Search bearer token during session initialization.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
SEARCH_ENDPOINT = os.environ["AZURE_AI_SEARCH_SERVICE_ENDPOINT"].rstrip("/")
KNOWLEDGE_BASE_NAME = os.environ.get(
    "AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME",
    "contoso-company-kb-minimal",
)
SEARCH_SCOPE = "https://search.azure.com/.default"
KNOWLEDGE_BASE_MCP_URL = (
    f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}"
    "/mcp?api-version=2026-05-01-preview"
)


@asynccontextmanager
async def open_knowledge_base_session():
    credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
    token = credential.get_token(SEARCH_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(
            KNOWLEDGE_BASE_MCP_URL,
            http_client=http_client,
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print(f"Foundry IQ MCP configured for {KNOWLEDGE_BASE_NAME}")

## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the minimal Foundry IQ knowledge base, including its detailed retrieval guidance and input schema.

In [ ]:
from rich.console import Console

console = Console()

async with open_knowledge_base_session() as session:
    tools_response = await session.list_tools()

knowledge_base_tools = tools_response.tools

for tool in knowledge_base_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

## Step 3: Retrieve from the minimal knowledge base

Call `knowledge_base_retrieve` with one to three concise search queries. The minimal-reasoning knowledge base returns extractive, source-attributable results rather than synthesizing a final answer.

In [ ]:
import json

queries = [
    "employee health plan benefits enrollment",
    "benefits enrollment deadlines",
]

async with open_knowledge_base_session() as session:
    result = await session.call_tool(
        "knowledge_base_retrieve",
        {"queries": queries},
    )

responses = [block.text for block in result.content if block.type == "text"]
for response in responses:
    console.print_json(data=json.loads(response))